In [1]:
# ==========================================================
# Notebook 04: Semantic Vector Search (FAISS)
# ==========================================================

import os
import pickle

import faiss
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
DATA_PATH = "data/processed/enterprise_corpus_acl.csv"

enterprise_df = pd.read_csv(DATA_PATH)

enterprise_df.head()

,source,title,content,author,doc_id,department,created_date,tags,acl_roles,security_level,owner_team
0,Slack,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...,Alice,DOC0001,Engineering,2026-01-11,"['enterprise-search', 'semantic-search']","['Engineering', 'Public']",Internal,Engineering
1,Slack,Model Training Update,The semantic search model has been retrained u...,Bob,DOC0002,Finance,2026-01-12,"['phoenix', 'kubernetes']","['Finance', 'Public']",Restricted,Finance
2,Confluence,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...,Charlie,DOC0003,HR,2026-01-13,"['semantic-search', 'enterprise-search']","['HR', 'Public']",Restricted,HR
3,Confluence,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...,Diana,DOC0004,Engineering,2026-01-14,"['fastapi', 'acl']","['Engineering', 'Public']",Internal,Engineering
4,Jira,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...,Eric,DOC0005,Engineering,2026-01-15,"['enterprise-search', 'kubernetes']","['Engineering', 'Public']",Internal,Engineering


In [3]:
enterprise_df["search_text"] = enterprise_df["title"] + " " + enterprise_df["content"]

In [ ]:
enterprise_df[["doc_id", "search_text"]].head()

,doc_id,search_text
0,DOC0001,Phoenix Deployment Discussion The Phoenix-2026...
1,DOC0002,Model Training Update The semantic search mode...
2,DOC0003,Phoenix Architecture Wiki Project Phoenix-2026...
3,DOC0004,Enterprise Search Design Hybrid search combine...
4,DOC0005,PHX-245 Database Migration Complete PostgreSQL...


In [5]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 2232d51b-c400-4932-bbad-22061d55acc7)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 81505645-a875-4494-ac57-b39cf8627685)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 2s [Retry 2/5].
d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=

In [6]:
document_texts = enterprise_df["search_text"].tolist()

document_embeddings = embedding_model.encode(
    document_texts, convert_to_numpy=True, show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
print("Embedding shape:")

print(document_embeddings.shape)

Embedding shape:
(6, 384)


In [8]:
document_embeddings = np.array(document_embeddings).astype("float32")

faiss.normalize_L2(document_embeddings)

In [9]:
embedding_dimension = document_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(document_embeddings)

In [10]:
print("Number of indexed documents:")

print(faiss_index.ntotal)

Number of indexed documents:
6


In [11]:
def semantic_search(query, model, index, dataframe, top_k=5):

    query_embedding = model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, ids = index.search(query_embedding, top_k)

    results = dataframe.iloc[ids[0]].copy()

    results["semantic_score"] = scores[0]

    return results.reset_index(drop=True)

In [12]:
query = "vector retrieval system"

semantic_results = semantic_search(
    query, embedding_model, faiss_index, enterprise_df, top_k=3
)

semantic_results[["doc_id", "title", "semantic_score"]]

,doc_id,title,semantic_score
0,DOC0004,Enterprise Search Design,0.449205
1,DOC0002,Model Training Update,0.245370
2,DOC0006,SEC-104 ACL Enhancement,0.191989


In [13]:
queries = [
    "vector retrieval",
    "semantic search",
    "database migration",
    "microservice architecture",
]

for query in queries:

    print("=" * 70)

    print(f"QUERY: {query}")

    output = semantic_search(
        query, embedding_model, faiss_index, enterprise_df, top_k=2
    )

    print(output[["title", "semantic_score"]])

    print()

QUERY: vector retrieval
                      title  semantic_score
0  Enterprise Search Design        0.430410
1     Model Training Update        0.207575

QUERY: semantic search
                      title  semantic_score
0     Model Training Update        0.619599
1  Enterprise Search Design        0.587106

QUERY: database migration
                           title  semantic_score
0     PHX-245 Database Migration        0.481465
1  Phoenix Deployment Discussion        0.428245

QUERY: microservice architecture
                           title  semantic_score
0      Phoenix Architecture Wiki        0.635807
1  Phoenix Deployment Discussion        0.150615



In [14]:
OUTPUT_PATH = "artifacts/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

faiss.write_index(faiss_index, OUTPUT_PATH + "enterprise_faiss.index")

print("FAISS index saved!")

FAISS index saved!


In [15]:
with open(OUTPUT_PATH + "document_embeddings.pkl", "wb") as file:

    pickle.dump(document_embeddings, file)

print("Embeddings saved!")

Embeddings saved!


In [16]:
loaded_index = faiss.read_index("artifacts/enterprise_faiss.index")

print("FAISS index loaded!")

FAISS index loaded!
